In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback

import numpy as np
from scipy.integrate import solve_ivp
from matplotlib import pyplot as plt

import torchphysics as tp

# Create Dataset

In [ ]:
def get_rhs(gamma, omega):
    """
    Parameters
    -------
    gamma : float
    """
    def rhs(t):
        return gamma * np.cos(omega*t)
    return rhs

def duffing(t, y, rhs, delta, alpha, beta):
    """
    Parameters
    -------
    t : float
        time step
    y : tuple(float, float)
        solution and derivative (y_t, v_t) at time step
    rhs : callable
        right hand side of duffing
    """
    x, x_dot = y
    y_dot = [
        x_dot, 
        -delta * x_dot - alpha * x - beta * x**3 + rhs(t),
        ]
    return y_dot

def calculate_duffing(rhs, delta, alpha, beta, eval_points, initial_condition):
    """
    Parameters
    -------
    rhs : callable
        rhs function of duffing
    delta : float
        damping coefficient
    alpha : float
        linear stiffness coefficient
    beta : float
        nonlinear stiffness coefficient
    eval_points : torch.Tensor
        Points at which duffing will be evaluated (shape: (n_points, 1))
    initial_condition : tuple(float, float)
        initial condition (y0, v0) for duffing
    
    Returns
    -------
    torch.Tensor
        solution of duffing (shape: (n_points, 1))
    """
    # solve_ivp expects shape [50]
    t_eval = eval_points.flatten()
    t_span = [ eval_points[0].item(), eval_points[-1].item()]
    sol = solve_ivp(duffing, t_span, initial_condition,
                    args=(rhs, delta, alpha, beta), t_eval=t_eval)
    # return torch.from_numpy(sol.y.reshape([len(t_eval), 2]))
    return torch.from_numpy(np.stack(sol.y, axis=-1))

In [ ]:
# Solver parameters
dataset_len = 50000
x0 = [0.0, 1.0] # inital position/speed
time_duration = 20
num_sensors = 100    # number of sensors for branch net
sensors = np.linspace(0, time_duration, num_sensors)
num_locations = 100 # number of locations for trunk net

# Parameters for the duffing equation
delta = 0.3
alpha = -1.0
beta = 1.0
omega = np.random.uniform(0.1, 2, dataset_len)
gamma = np.random.uniform(0, 1, dataset_len)

In [ ]:
u_data_tensor = np.zeros((dataset_len, num_sensors, 1))
t_data_tensor = np.zeros((dataset_len, num_locations, 1))
f_data_tensor = np.zeros((dataset_len, num_locations, 2))

for idx, (ga, om) in enumerate(zip(gamma, omega)):
    locations = np.linspace(0, time_duration, num_locations)
    rhs = get_rhs(ga, om)
    u_data_tensor[idx] = rhs(sensors)[:, None]
    t_data_tensor[idx] = locations[:, None]
    f_data_tensor[idx] = calculate_duffing(rhs, delta, alpha, beta, locations, x0)
u_data_tensor = torch.tensor(u_data_tensor, dtype=torch.float32)
t_data_tensor = torch.tensor(t_data_tensor, dtype=torch.float32)
f_data_tensor = torch.tensor(f_data_tensor, dtype=torch.float32)
print(f'u: {u_data_tensor.shape}\n', f'y: {t_data_tensor.shape}\n', f'f: {f_data_tensor.shape}\n')

In [ ]:
class DeepONetDataset(Dataset):
    def __init__(self, y_data, u_data, f_data):
        self.y_data = y_data      # shape: (N, 10, 1)
        self.u_data = u_data      # shape: (N, 50, 1)
        self.f_data = f_data    # shape: (N, 10, 1)

    def __len__(self):
        return self.y_data.shape[0]

    def __getitem__(self, idx):
        return {
            'y': self.y_data[idx],       # (10, 1)
            'u': self.u_data[idx],       # (50, 1)
            'f': self.f_data[idx]  # (10, 1)
        }

In [ ]:
dataset = DeepONetDataset(t_data_tensor, u_data_tensor, f_data_tensor)

# Plot Dataset

In [ ]:
def plot_duffing_solution(
        dataset: DeepONetDataset,
        model: tp.DeepONetV2 | None = None,
        idx: int = 0,
        ax: plt.Axes = None
        ) -> tuple[plt.Figure, plt.Axes]:
    """Plot the solution of the Duffing equation for a given index.

    Note:
    This function only plots first dimension trunk input.

    Parameters
    ----------
    dataset : DeepONetDataset
        The dataset containing the input and output data.
    model : DeepONetV2
        The trained DeepONet model.
    idx : int
        The index of the data point to plot.
    ax : plt.Axes
        The axes on which to plot the data.
    
    Returns
    -------
    tuple[plt.Figure, plt.Axes]
        The figure and axes of the plot.
    """
    if ax is None:
        _, ax = plt.subplots()
    
    data = dataset[idx]
    u = data['u']
    t = data['y']
    f = data['f']

    for dim in range(f.shape[-1]):
        ax.plot(t[:, 0], f[:, dim], label=f'$f_{dim}$')

    if model is not None:
        with torch.no_grad():
            # Add batch for input and remove batch for output
            prediction = model(t[None], u[None])[0]

        for dim in range(prediction.shape[-1]):
            label = '$f^{~pred}_{%s}$'.replace('%s', str(dim))
            ax.plot(t[:, 0], prediction[:, dim].numpy(), label=label, linestyle='--')
    
    ax.legend()
    ax.grid()
    ax.set_title(f'Solution at Index {idx}')
    return ax.get_figure(), ax

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(8, 8))
axs = axs.flatten()
for i, idx in enumerate(np.random.choice(len(dataset), size=len(axs), replace=False)):
    plot_duffing_solution(dataset, idx=idx, ax=axs[i])
for ax in axs[:-1]:
    ax.legend().remove()
fig.tight_layout()

# DeepONet

In [ ]:
class CustomFCNBranchNet(nn.Module):
    def __init__(self):
        super(CustomFCNBranchNet, self).__init__()
        self.fc1 = nn.Linear(100, 80)
        self.fc2 = nn.Linear(80, 60)
        self.fc3 = nn.Linear(60, 40)
        self.activation = nn.Tanh()
    
    def forward(self, x):
        # -> 1, 100, 1 (batch, num_locations, channels)
        x = x.squeeze(-1)
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        x = self.activation(x)
        x = self.fc3(x)
        x = self.activation(x)
        # -> 1, 100, 100 (batch, num_locations, channels)
        return x

class CustomFCNTrunkNet(nn.Module):
    def __init__(self):
        super(CustomFCNTrunkNet, self).__init__()
        self.fc1 = nn.Linear(1, 10)
        self.fc2 = nn.Linear(10, 20)
        self.fc3 = nn.Linear(20, 20)
        self.activation = nn.Tanh()

    def forward(self, x):
        # -> 1, 100, 1 (batch, num_locations, channels)
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        x = self.activation(x)
        x = self.fc3(x)
        x = self.activation(x)
        # -> 1, 100, 100 (batch, num_locations, channels)
        return x

branch = tp.BranchNetV2(CustomFCNBranchNet(), 1, 2, 10, 100, True)
trunk = tp.TrunkNetV2(CustomFCNTrunkNet(), 1, 2, 100, 10, True)
model = tp.DeepONetV2(trunk, branch)
model

### Overview of Dimensions
- batch: Dataset batch size
- num_sensors: number of sensors for u
- dim: dimension of vector (e.g. [x, dx])
- num_locations: number of values where TrunkNet will be evaluated
- channels: latent space

In [ ]:
data = dataset[0]
u = data['u'][None]     # Add batch dimension for input
y = data['y'][None]
f = data['f']
b_out = branch(u)
t_out = trunk(y)
m_out = model(y, u)
print(f"u: \t\t{u.shape} \t(batch, num_sensors, input_dim)\n"
      f"y: \t\t{y.shape}  \t(batch, num_locations, input_dim)\n"
      f"BranchNet: \t{b_out.shape} \t\t(batch, output_dim, channels)\n"
      f"TrunkNet: \t{t_out.shape} \t(batch, num_locations, output_dim, channels)\n"
      f"DeepONet: \t{m_out.shape} \t(batch, num_locations, output_dim)\n")

# Train Model

In [ ]:
class DeepONetLightning(pl.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.loss_fn = torch.nn.MSELoss()

    def forward(self, y, u):
        return self.model(y, u)

    def training_step(self, batch, batch_idx):
        y, u, target = batch['y'], batch['u'], batch['f']
        output = self(y, u)
        loss = self.loss_fn(output, target)
        self.log('train_loss', loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        y, u, target = batch['y'], batch['u'], batch['f']
        output = self(y, u)
        val_loss = self.loss_fn(output, target)
        self.log('val_loss', val_loss, prog_bar=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-2)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'frequency': 1
            }
        }


class LossHistory(Callback):
    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        # Get the latest logged train loss (averaged over the epoch)
        train_loss = trainer.callback_metrics.get("train_loss")
        if train_loss is not None:
            self.train_losses.append(train_loss.cpu().item())

    def on_validation_epoch_end(self, trainer, pl_module):
        # Get the latest logged val loss (averaged over the epoch)
        val_loss = trainer.callback_metrics.get("val_loss")
        if val_loss is not None:
            self.val_losses.append(val_loss.cpu().item())

In [ ]:
dataset = DeepONetDataset(t_data_tensor, u_data_tensor, f_data_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

loss_history = LossHistory()
lightning_model = DeepONetLightning(model)
trainer = pl.Trainer(max_epochs=8,
                     accelerator='cuda' if torch.cuda.is_available() else 'cpu',
                     devices=1 if torch.cuda.is_available() else None,
                     logger=False,
                     callbacks=[loss_history],
                     num_sanity_val_steps=np.ceil(len(val_dataset)/batch_size),
                     benchmark=True,
                     enable_checkpointing=False)

In [ ]:
trainer.fit(lightning_model, train_loader, val_loader)

# Results

In [ ]:
fig, ax = plt.subplots()
ax.plot(loss_history.val_losses, label='Validation Loss')
ax.plot(range(1, len(loss_history.train_losses)+1), loss_history.train_losses,
        label='Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.set_yscale('log')
ax.grid()

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(8, 8))
axs = axs.flatten()
for i, idx in enumerate(np.random.choice(len(val_dataset), size=len(axs), replace=False)):
    plot_duffing_solution(val_dataset, model, idx=idx, ax=axs[i])
for ax in axs[:-1]:
    ax.legend().remove()
fig.tight_layout()

In [ ]:
import time

idx = 0
data = dataset[idx]
u = data['u']
t = data['y']
f = data['f']

# --- Numeric solution timing ---
start = time.time()
rhs = get_rhs(gamma[idx].item(), omega[idx].item())
numeric_sol = calculate_duffing(rhs, delta, alpha, beta, t, x0)
numeric_time = time.time() - start

# --- DeepONet prediction timing ---
with torch.no_grad():
    start = time.time()
    deeponet_pred = model(t[None], u[None])[0]
    deeponet_time = time.time() - start

print(f"Numeric solution time: {numeric_time:.6f} seconds")
print(f"DeepONet prediction time: {deeponet_time:.6f} seconds")